# Mobile Money Agent Network: Audit and Cleanup

**Analyst:** EUNICE LIVINGPROOF | **Cohort:** 1MILDEVS

The legacy transaction extract is unreliable. This notebook audits it, cleans it with
`clean_agent_data()` from `cleaning_pipeline.py`, merges it with the agent register
and produces summary tables for the regional manager.

In [1]:
import os

import numpy as np
import pandas as pd

from cleaning_pipeline import clean_agent_data

os.makedirs("outputs", exist_ok=True)

## 1. Load and audit the raw data

In [2]:
txns = pd.read_csv("agent_transactions.csv")
agents = pd.read_csv("agents.csv")
raw_backup = txns.copy()                     # the raw data is never overwritten

# Second read that keeps blanks and the text "N/A" as text, to tell them apart
raw_text = pd.read_csv("agent_transactions.csv", dtype=str, keep_default_na=False)

print("Transactions:", txns.shape, "| Agents:", agents.shape)
display(txns.head())
txns.info()

Transactions: (612, 5) | Agents: (20, 3)


,txn_id,agent_id,txn_type,amount,txn_date
0,TX00001,AG009,deposit,353.0,2026-03-03
1,TX00002,AG020,deposit,1238.0,2026-04-27
2,TX00003,AG008,DEPOSIT,9129.0,2026-04-29
3,TX00004,AG009,deposit,652.0,2026-04-24
4,TX00005,AG009,withdrawal,478.0,2026-04-17


<class 'pandas.DataFrame'>
RangeIndex: 612 entries, 0 to 611
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   txn_id    612 non-null    str    
 1   agent_id  612 non-null    str    
 2   txn_type  612 non-null    str    
 3   amount    590 non-null    float64
 4   txn_date  612 non-null    str    
dtypes: float64(1), str(4)
memory usage: 24.0 KB


In [3]:
issues = [
    ("Duplicate txn_id rows", "Uniqueness", "txn_id",
     int(txns.duplicated("txn_id").sum())),
    ("Exact duplicate rows (all columns identical)", "Uniqueness", "(all columns)",
     int(txns.duplicated().sum())),
    ("txn_type not exactly 'Deposit' or 'Withdrawal' (case and space variants)", "Consistency", "txn_type",
     int((~txns["txn_type"].isin(["Deposit", "Withdrawal"])).sum())),
    ("Blank amount", "Completeness", "amount",
     int((raw_text["amount"].str.strip() == "").sum())),
    ("Text 'N/A' in amount", "Completeness", "amount",
     int((raw_text["amount"].str.strip() == "N/A").sum())),
    ("Amount of exactly 250,000 (repeated suspicious value)", "Accuracy", "amount",
     int((txns["amount"] == 250000).sum())),
    ("Dates written with '/' instead of '-'", "Consistency", "txn_date",
     int(txns["txn_date"].str.contains("/", regex=False).sum())),
    ("agent_id missing from the agent register", "Validity", "agent_id",
     int((~txns["agent_id"].isin(agents["agent_id"])).sum())),
]
audit = pd.DataFrame(issues, columns=["issue", "dimension", "column", "rows_affected"])
display(audit)

,issue,dimension,column,rows_affected
0,Duplicate txn_id rows,Uniqueness,txn_id,12
1,Exact duplicate rows (all columns identical),Uniqueness,(all columns),11
2,txn_type not exactly 'Deposit' or 'Withdrawal'...,Consistency,txn_type,384
3,Blank amount,Completeness,amount,15
4,Text 'N/A' in amount,Completeness,amount,7
5,"Amount of exactly 250,000 (repeated suspicious...",Accuracy,amount,5
6,Dates written with '/' instead of '-',Consistency,txn_date,5
7,agent_id missing from the agent register,Validity,agent_id,8


### Audit notes

- pandas reads the text `N/A` as missing by default, so a plain `isna()` count of **22** missing amounts mixes two problems: **15** blank cells and **7** cells containing the text `N/A`. The second read with `keep_default_na=False` separates them.
- The raw file has **612** rows but only **600** distinct `txn_id` values, so **12** rows are repeats, of which **11** are exact copies of the whole row.
- The exact amount 250,000 appears **5** times. Real amounts rarely repeat to the shilling, so it looks like a placeholder or a system limit.

## 2. Clean with the pipeline

In [4]:
clean_df, log = clean_agent_data(txns, agents)
for line in log:
    print("-", line)
print()
print("Raw rows:", len(raw_backup), "| Clean rows:", len(clean_df))

- Uniqueness: removed 12 duplicate txn_id rows (11 were exact copies); 600 rows remain
- Consistency: standardised txn_type on 376 rows (0 could not be mapped)
- Validity: converted amount to numeric (20 missing or invalid) and fixed 5 dates written with '/'
- Outliers: flagged 40 amounts above the upper fence of KES 6,084 (kept, not deleted)
- Completeness: imputed 20 amounts with the median of the same txn_type, and flagged them
- Merge: 600 rows in, 600 rows out; 7 transactions reference unknown agents (county set to 'Unknown', rows kept)

Raw rows: 612 | Clean rows: 600


## 3. Uniqueness and consistency (Tasks 2 and 3)

In [5]:
print("txn_id unique after cleaning:", clean_df["txn_id"].is_unique)
display(clean_df["txn_type"].value_counts())
print("Raw spellings that were found:", sorted(raw_backup["txn_type"].unique().tolist()))

txn_id unique after cleaning: True


txn_type
Deposit       373
Withdrawal    227
Name: count, dtype: int64

Raw spellings that were found: ['DEPOSIT ', 'Deposit', 'Withdrawal', 'deposit', 'withdrawal']


## 4. Validity, types and completeness (Tasks 4 and 5)

In [6]:
print(clean_df.dtypes)
print("txn_date range:", clean_df["txn_date"].min().date(), "to", clean_df["txn_date"].max().date())
print("Missing values left:", int(clean_df[["amount", "txn_date", "txn_type"]].isna().sum().sum()))

imputed = clean_df[clean_df["amount_imputed"]]
print("Imputed rows:", len(imputed))
display(imputed.groupby("txn_type")["amount"].agg(["count", "first"]).rename(columns={"first": "filled_with"}))

txn_id                       str
agent_id                     str
txn_type                     str
amount                   float64
txn_date          datetime64[us]
amount_imputed              bool
is_outlier                  bool
county                       str
onboarded         datetime64[us]
agent_known                 bool
dtype: object
txn_date range: 2026-03-01 to 2026-04-29
Missing values left: 0
Imputed rows: 20


,count,filled_with
txn_type,,
Deposit,13,1803.5
Withdrawal,7,1750.5


### Decision: missing and invalid amounts

**20** transactions (3.3% of 600) had a missing or invalid amount (15 blank and 7 written as `N/A`, before duplicates were removed).

- **Choice:** keep the rows and fill each amount with the **median of its own transaction type** (Deposit KES 1,804, Withdrawal KES 1,750). The median is not pulled upwards by the 250,000 values.
- **Why not drop them:** each row still shows that a transaction happened, with an agent, a date and a type, and dropping them would undercount activity.
- **Why not trust the filled values:** they are guesses. The `amount_imputed` flag stays in the data, and all value totals below use only rows with a recorded amount, so no invented money reaches the totals.
- **Why not use the mean:** it would be inflated by the outliers.

## 5. Outliers (Task 6)

In [7]:
observed = clean_df.loc[~clean_df["amount_imputed"], "amount"]
q1, q3 = observed.quantile([0.25, 0.75])
iqr = q3 - q1
upper = q3 + 1.5 * iqr
print(f"Q1 = {q1:,.0f} | Q3 = {q3:,.0f} | IQR = {iqr:,.0f} | upper fence = {upper:,.0f}")
print("Flagged outliers:", int(clean_df["is_outlier"].sum()))

flagged = clean_df[clean_df["is_outlier"]]
print("\nMost common flagged amounts:")
print(flagged["amount"].value_counts().head())
display(flagged.sort_values("amount", ascending=False).head(8)[["txn_id", "agent_id", "txn_type", "amount", "txn_date"]])

Q1 = 1,077 | Q3 = 3,080 | IQR = 2,003 | upper fence = 6,084
Flagged outliers: 40

Most common flagged amounts:
amount
250000.0    5
9129.0      1
8507.0      1
8035.0      1
6566.0      1
Name: count, dtype: int64


,txn_id,agent_id,txn_type,amount,txn_date
424,TX00425,AG005,Withdrawal,250000.0,2026-03-02
463,TX00464,AG009,Withdrawal,250000.0,2026-04-11
315,TX00316,AG014,Deposit,250000.0,2026-04-01
594,TX00595,AG016,Deposit,250000.0,2026-03-11
306,TX00307,AG012,Deposit,250000.0,2026-03-02
183,TX00184,AG006,Deposit,28847.0,2026-03-08
222,TX00223,AG011,Deposit,22316.0,2026-03-02
573,TX00574,AG017,Deposit,16605.0,2026-03-16


### Recommendation: the KES 250,000 values

The IQR method gives an upper fence of **KES 6,084** and flags **40** amounts. **5** of them are exactly KES 250,000, worth KES 1,250,000 together, or 47.1% of all recorded value. The other 35 flagged amounts are ordinary large transactions in a naturally skewed distribution.

**Recommendation:**
- Do **not** delete or silently cap them. Send the 5 transaction IDs to the data owner and ask whether 250,000 is a per-transaction limit, a default value, or a real large transaction.
- Until the answer arrives, keep them in the data with `is_outlier = True`, and report totals both with and without them. Use medians for typical-transaction statements.
- If they are placeholders, treat them as missing and impute them. If they are real, they are the most important transactions for the business to review.

## 6. Merge validation (Task 7)

In [8]:
rows_after_dedup = len(raw_backup.drop_duplicates("txn_id"))
print("Rows before the merge (after removing duplicates):", rows_after_dedup)
print("Rows after the merge:", len(clean_df))
print("Row count preserved:", rows_after_dedup == len(clean_df))
print("Transactions with unknown agents:", int((~clean_df["agent_known"]).sum()))
display(clean_df.loc[~clean_df["agent_known"], "agent_id"].value_counts())
display(clean_df["county"].value_counts())

Rows before the merge (after removing duplicates): 600
Rows after the merge: 600
Row count preserved: True
Transactions with unknown agents: 7


agent_id
AG099    7
Name: count, dtype: int64

county
Nairobi    180
Mombasa    133
Kisumu     111
Nakuru      99
Kiambu      70
Unknown      7
Name: count, dtype: int64

### How unknown agents were handled

- Join type: **left join** from transactions to the agent register, with `validate="many_to_one"` and `indicator=True`. An inner join would silently drop transactions.
- Row count before and after the merge: **600** and **600**, so nothing was lost or multiplied.
- **8** raw transactions (7 after removing duplicates) reference agent AG099, which is not in the register.
- **Handling:** the rows are kept, their county is set to `Unknown`, and `agent_known` is `False`. They carry KES 27,156 of recorded value, so removing them would hide real money. The agent ID should be checked with the register owner, because it may be an unregistered agent or a typing error.

## 7. Summaries (Task 8)

Value summaries use only rows with a **recorded** amount, because imputed amounts would
add invented money to the totals. `transactions` therefore counts rows with a recorded amount.

In [9]:
recorded = clean_df[~clean_df["amount_imputed"]]

county_summary = (
    recorded.groupby(["county", "txn_type"])
    .agg(total_value=("amount", "sum"),
         transactions=("txn_id", "count"),
         avg_value=("amount", "mean"))
    .round(0)
    .reset_index()
)
display(county_summary)

,county,txn_type,total_value,transactions,avg_value
0,Kiambu,Deposit,334737.0,43,7785.0
1,Kiambu,Withdrawal,75274.0,26,2895.0
2,Kisumu,Deposit,431091.0,71,6072.0
3,Kisumu,Withdrawal,74598.0,36,2072.0
4,Mombasa,Deposit,141880.0,68,2086.0
5,Mombasa,Withdrawal,368592.0,61,6042.0
6,Nairobi,Deposit,536484.0,114,4706.0
7,Nairobi,Withdrawal,154930.0,58,2671.0
8,Nakuru,Deposit,181337.0,58,3126.0
9,Nakuru,Withdrawal,326818.0,38,8600.0


In [10]:
recorded = recorded.assign(
    week_start=recorded["txn_date"].dt.to_period("W").dt.start_time.dt.strftime("%Y-%m-%d")
)
weekly = pd.pivot_table(recorded, index="week_start", columns="county",
                        values="amount", aggfunc="sum", fill_value=0).round(0)
display(weekly)

county,Kiambu,Kisumu,Mombasa,Nairobi,Nakuru,Unknown
week_start,,,,,,
2026-02-23,0.0,1893.0,6759.0,7046.0,967.0,0.0
2026-03-02,18317.0,287933.0,283893.0,68967.0,56574.0,0.0
2026-03-09,265995.0,23186.0,28701.0,51084.0,28730.0,0.0
2026-03-16,22054.0,24572.0,23368.0,41331.0,39795.0,1639.0
2026-03-23,11504.0,27077.0,17050.0,39597.0,17222.0,0.0
2026-03-30,22609.0,41077.0,38344.0,282931.0,18145.0,8507.0
2026-04-06,15030.0,15341.0,45316.0,55010.0,276609.0,9824.0
2026-04-13,12265.0,25863.0,26730.0,54656.0,24376.0,0.0
2026-04-20,24507.0,37439.0,16687.0,61634.0,30862.0,1877.0


## 8. Save the clean data

In [11]:
export = clean_df.copy()
export["txn_date"] = export["txn_date"].dt.strftime("%Y-%m-%d")
export["onboarded"] = export["onboarded"].dt.strftime("%Y-%m-%d")
export.to_csv("outputs/agent_transactions_clean.csv", index=False)

check = pd.read_csv("outputs/agent_transactions_clean.csv")
print("Saved:", check.shape)
print("txn_id unique:", check["txn_id"].is_unique)
print("txn_type values:", sorted(check["txn_type"].unique()))
print("Columns:", check.columns.tolist())

Saved: (600, 10)
txn_id unique: True
txn_type values: ['Deposit', 'Withdrawal']
Columns: ['txn_id', 'agent_id', 'txn_type', 'amount', 'txn_date', 'amount_imputed', 'is_outlier', 'county', 'onboarded', 'agent_known']


## 9. Data Quality Report

**Scope.** The extract has 612 rows and 600 unique transactions after cleaning, covering 2026-03-01 to 2026-04-29 (60 days), with 20 agents in the register. Every number below was calculated by code.

**1. Completeness.** 22 amounts were missing (15 blank, 7 as the text `N/A`). They were flagged in `amount_imputed` and filled with the median of the same transaction type. Value totals exclude imputed rows.

**2. Validity.** 5 dates used `/` instead of `-` and were converted to proper datetimes. `amount` arrived as text and was converted to numbers, with anything unreadable turned into a flagged missing value. 8 rows referenced `AG099`, an agent that does not exist in the register.

**3. Accuracy.** The amount 250,000 appears 5 times, which is unlikely for real transactions. These are flagged with `is_outlier` and kept pending confirmation from the data owner. Accuracy cannot be fully checked without the source system.

**4. Consistency.** 384 rows spelled `txn_type` with different case or trailing spaces (`DEPOSIT `, `withdrawal`, and so on). They were standardised to exactly `Deposit` and `Withdrawal`. Dates were also written in two formats.

**5. Uniqueness.** 12 rows repeated an existing `txn_id` (11 were exact copies). They were removed, keeping the first occurrence, and `txn_id` is now unique.

**6. Timeliness.** The data covers only 60 days. 0 transactions are dated before their agent's onboarding date. The extract's age should be confirmed with the data owner before it is used for a current decision.

**Decisions to confirm with the business.** The meaning of the 250,000 amounts, the identity of agent AG099, and whether imputed amounts should instead be excluded.

**Privacy (Kenya Data Protection Act, 2019).** This extract holds no personal data, but if it also contained customer phone numbers, then the phone number, the customer name, any national ID number and the transaction history linked to them would all count as personal data. Mobile-money records can reveal income, spending habits and location, so they deserve extra care. I would follow the Act's principles: use only the fields the analysis needs (data minimisation), so phone numbers would be dropped unless essential; replace identifiers with a salted hash (pseudonymisation) so customers can still be counted without being identified; mask any phone number shown in a report (for example `0712****78`); publish only aggregates and suppress groups of fewer than five people; keep the salt and the raw file out of GitHub using `.gitignore`; and use the data only for the purpose it was collected for, keeping it no longer than needed.